# Proyecto II: Detección de anomalías industriales mediante destilado de modelos y autoencoders

**Curso:** Inteligencia Artificial, Escuela de Ingeniería en Computación, Instituto Tecnológico de Costa Rica

**Integrantes:**
- Sebastián Chacón - 2023221908
- Daniel Pulido - 2019196502
- Sebastián Muñoz - 2023090681

**Notebook 2 de 4: modelos A y B.** Entrenar el clasificador CNN con la estructura de ResNet-18 hasta `conv3_x`, desde cero (modelo A) y por destilado teacher student a partir de un ResNet-18 completo (modelo B), con al menos 3 combinaciones de hiperparámetros cada uno.

El proyecto se divide en cuatro notebooks que comparten la configuración de `conf/` y el código de `src/`:
1. `datos.ipynb`: exploración de MVTec AD y `LightningDataModule`.
2. `modelos_ab.ipynb`: modelo A (CNN desde cero) y modelo B (CNN destilada).
3. `modelo_c.ipynb`: modelo C (autoencoder U-Net).
4. `anomalias.ipynb`: detección de anomalías con embeddings y análisis con DBSCAN.

## Contenido
1. Configuración y entorno
2. Modelo A, CNN desde cero
3. Modelo B, CNN destilada
4. Comparación de los modelos A y B

## 1. Configuración y entorno

### 1.1 Imports y configuración con Hydra
Las clases (DataModule, modelos y utilidades de entrenamiento) están en `src/` como scripts auxiliares, y este notebook las importa y ejecuta. Para eso se agrega la raíz del repositorio a `sys.path`. Con `autoreload` los cambios en `src/` se cargan sin reiniciar el kernel.

El decorador `@hydra.main` está pensado para scripts que se ejecutan desde la terminal y no funciona dentro de Jupyter. Por eso `load_config` usa la API de composición de Hydra (`initialize_config_dir` y `compose`), que es la que recomienda su documentación para notebooks. Recibe una lista de overrides, por ejemplo `["+experiment=cnn_scratch/lr3e-4"]`, y así cada corrida varía sus hiperparámetros sin tocar los archivos YAML.

In [1]:
# Imports, repository root and Hydra configuration
%load_ext autoreload
%autoreload 2

import logging
import sys
import warnings
from pathlib import Path

# Hide noise that does not affect the results: torch's "triton not found" (no triton on Windows)
logging.getLogger("torch.utils.flop_counter").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# The notebook lives in notebooks/, the repository root is its parent folder
PROJECT_ROOT = Path.cwd().resolve().parent
assert (PROJECT_ROOT / "conf").is_dir(), "Run the notebook from the notebooks/ folder"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils import load_config, resolve_path

cfg = load_config()
print(OmegaConf.to_yaml(cfg))

data:
  _target_: src.data.MVTecDataModule
  root: data/mvtec_ad
  categories:
  - carpet
  - leather
  - tile
  - wood
  - bottle
  - cable
  - capsule
  - hazelnut
  - metal_nut
  - pill
  image_size: 224
  batch_size: 32
  num_workers: 0
  cache_dir: ${paths.output_dir}/cache
  val_split: 0.2
  seed: ${seed}
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
model:
  _target_: src.models.cnn.CNNClassifierModule
  name: cnn_scratch
  num_classes: ${len:${data.categories}}
  head:
    hidden_dim: 256
    dropout: 0.3
  optimizer:
    _target_: torch.optim.AdamW
    _partial_: true
    lr: 0.001
    weight_decay: 0.0001
  scheduler: null
trainer:
  _target_: lightning.Trainer
  max_epochs: 50
  accelerator: auto
  devices: 1
  precision: 16-mixed
  log_every_n_steps: 10
  enable_progress_bar: false
callbacks:
  early_stopping:
    _target_: lightning.pytorch.callbacks.EarlyStopping
    monitor: val/loss
    mode: min
    patience: 5
    min_delta: 0.0
  checkpoi

### 1.2 Semilla y GPU
Se fija la semilla de Python, NumPy y PyTorch para que los resultados sean reproducibles, y se verifica que la GPU esté disponible.

In [2]:
# Seed, GPU and library versions
L.seed_everything(cfg.seed)
# Lets float32 matrix multiplications use the GPU Tensor Cores (TF32): faster, negligible precision change
torch.set_float32_matmul_precision("high")

print("PyTorch", torch.__version__, "| Lightning", L.__version__, "| wandb", wandb.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB de VRAM")

Seed set to 42


PyTorch 2.14.1+cu130 | Lightning 2.6.6 | wandb 0.31.2
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060, 8.0 GB de VRAM


## 2. Modelo A, CNN desde cero

Código en `src/models/cnn.py`, configuración en `conf/model/cnn_scratch.yaml` y combinaciones en `conf/experiment/cnn_scratch/`.

### 2.1 Backbone: ResNet-18 hasta `conv3_x`

*Pendiente.*

### 2.2 Clasificador y `get_embedding`

*Pendiente.*

### 2.3 `LightningModule`

*Pendiente.*

### 2.4 Combinaciones de hiperparámetros

*Pendiente.*

### 2.5 Entrenamiento

*Pendiente.*

### 2.6 Resultados y análisis de overfitting

*Pendiente.*

## 3. Modelo B, CNN destilada

Código en `src/models/distill.py`, configuración en `conf/model/cnn_distill.yaml` y combinaciones en `conf/experiment/cnn_distill/`.

### 3.1 Teacher: ResNet-18 preentrenado

*Pendiente.*

### 3.2 Técnica de destilado

*Pendiente.*

### 3.3 `LightningModule`

*Pendiente.*

### 3.4 Combinaciones de hiperparámetros

*Pendiente.*

### 3.5 Entrenamiento

*Pendiente.*

### 3.6 Resultados y análisis de overfitting

*Pendiente.*

## 4. Comparación de los modelos A y B

Parámetros, tiempo de inferencia y curvas de entrenamiento y validación del teacher, el modelo A y el modelo B.

*Pendiente.*